# NETRA Phase 3 — RoBERTa Tier-2 Escalation Training (GPU EDITION)

**Project:** NETRA — The AI Eye Against Phishing  
**Model:** RoBERTa-base (125M parameters, 512 token context)  
**Target Hardware:** NVIDIA RTX 5050 (8GB VRAM) / RTX 3050 / RTX 40-series / Google Colab (T4/A100)  
**Task:** 3-Class Deep Escalation Classification (`LEGITIMATE = 0`, `SUSPICIOUS = 1`, `PHISHING = 2`) with Authentication Header Fusion (SPF/DKIM/DMARC) + Tier-1 Confidence Integration.

---

### What Tier-2 Does in the NETRA Two-Tier Architecture:
1. **Tier-1 (DistilBERT, 66M params, 256 tokens):** Runs locally in ~180ms. Handles clear Legitimate and clear Phishing emails with high confidence.
2. **Escalation Policy:** When Tier-1 is uncertain (`SUSPICIOUS` verdict, `PHISHING` confidence < 0.70, or `LEGITIMATE` with threat signals triggered), the email is escalated to **Tier-2**.
3. **Tier-2 (RoBERTa-base, 125M params, 512 tokens):** Ingests the full 512-token body, 10 SPF/DKIM/DMARC authentication signals, and Tier-1 confidence, providing 3-class precision and Integrated Gradients (XAI) token attribution.


## Section 1: Check Local GPU & Environment

Verifies your NVIDIA CUDA GPU, VRAM capacity, and PyTorch configuration. Optimized for 8GB VRAM (e.g. RTX 5050).


In [ ]:
import os
import sys
import torch

print(f"Python Version    : {sys.version.split()[0]}")
print(f"PyTorch Version   : {torch.__version__}")
print(f"CUDA Available    : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device = torch.device("cuda")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU Device Name   : {props.name}")
    print(f"Total VRAM        : {props.total_memory / (1024**3):.2f} GB")
    print(f"Compute Capability: {props.major}.{props.minor}")
    # Enable TF32 for Ampere / Ada Lovelace / Blackwell architectures
    if props.major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        print("TF32 Acceleration : Enabled")
else:
    device = torch.device("cpu")
    print("WARNING: CUDA not detected. Training will run on CPU (significantly slower).")


## Section 2: Workspace & Path Setup

Automatically discovers the NETRA project root and ensures all necessary directories exist.


In [ ]:
from pathlib import Path

# Automatically detect project root whether running from repo root or notebooks/
CURRENT_DIR = Path(".").resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR       = PROJECT_ROOT / "data" / "processed"
RAW_DIR        = PROJECT_ROOT / "data" / "raw"
MODELS_DIR     = PROJECT_ROOT / "ml" / "models"
EVAL_DIR       = PROJECT_ROOT / "ml" / "evaluation"

UNIFIED_CSV    = DATA_DIR / "unified.csv"
TIER2_CSV      = DATA_DIR / "tier2_train.csv"

# Tier-1 Model Paths (used to generate escalation set if needed)
TIER1_MODEL    = MODELS_DIR / "distilbert_tier1.pt"
TIER1_CONFIG   = MODELS_DIR / "distilbert_config.json"
TIER1_TOK      = MODELS_DIR / "distilbert_tokenizer"

# Tier-2 Output Paths
TIER2_MODEL    = MODELS_DIR / "roberta_tier2.pt"
TIER2_CONFIG   = MODELS_DIR / "roberta_tier2_config.json"
TIER2_TOK      = MODELS_DIR / "roberta_tier2_tokenizer"
TIER2_METRICS  = EVAL_DIR / "roberta_tier2_metrics.json"

for d in [DATA_DIR, RAW_DIR, MODELS_DIR, EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"NETRA Project Root : {PROJECT_ROOT}")
print(f"Models Directory   : {MODELS_DIR}")
print(f"Evaluation Dir     : {EVAL_DIR}")
print(f"Tier-1 Weights     : {'Found' if TIER1_MODEL.exists() else 'Missing'}")
print(f"Unified CSV        : {'Found' if UNIFIED_CSV.exists() else 'Missing'}")
print(f"Tier-2 CSV         : {'Found' if TIER2_CSV.exists() else 'Missing (Will generate)'}")


## Section 3: Load or Generate Tier-2 Escalation Dataset

Tier-2 is trained specifically on **escalated records**:
- Emails falling within Tier-1's ambiguity corridor (`risk_score` between 0.03 and 0.50)
- Emails with low Tier-1 confidence (< 0.70)
- Emails with conflicting security signals (e.g. SPF/DKIM fail but low text score)

If `data/processed/tier2_train.csv` already exists, we load it directly. If not, this step runs Tier-1 inference across `unified.csv` to generate it automatically.


In [ ]:
import pandas as pd
import numpy as np

if TIER2_CSV.exists():
    print(f"Loading existing Tier-2 dataset from: {TIER2_CSV}")
    df_tier2 = pd.read_csv(TIER2_CSV)
else:
    print("tier2_train.csv not found. Generating from unified.csv using Tier-1 model...")
    assert UNIFIED_CSV.exists(), f"unified.csv not found at {UNIFIED_CSV}! Please place unified.csv in data/processed/"
    assert TIER1_MODEL.exists(), f"distilbert_tier1.pt not found at {TIER1_MODEL}!"
    
    # Run dataset generator logic
    from ml.generate_tier2_dataset import generate_tier2_dataset
    generate_tier2_dataset(UNIFIED_CSV, TIER2_CSV)
    df_tier2 = pd.read_csv(TIER2_CSV)

print(f"\nTotal Tier-2 Records: {len(df_tier2):,}")
print("\nClass Distribution (tier2_label):")
label_map = {0: "LEGITIMATE", 1: "SUSPICIOUS", 2: "PHISHING"}
counts = df_tier2["tier2_label"].value_counts().sort_index()
for lbl, count in counts.items():
    pct = count / len(df_tier2) * 100
    print(f"  Class {lbl} ({label_map.get(int(lbl), 'Unknown')}): {count:,} ({pct:.1f}%)")

print("\nDataset Splits:")
print(df_tier2["split"].value_counts())
df_tier2.head(3)


## Section 4: Tokenization & PyTorch Dataset Setup

We use `roberta-base` with:
- **Max length:** 512 tokens (capturing full email context)
- **10 Header Auth Features:** SPF (pass, fail, none), DKIM (pass, fail, none), DMARC (pass, fail, none), and domain match
- **Tier-1 Confidence:** Scalar probability from Tier-1 model


In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

TOKENIZER_NAME = "roberta-base"
MAX_LENGTH = 512
BATCH_SIZE = 16  # Optimal for 8GB VRAM (RTX 5050)

print(f"Loading RoBERTa tokenizer: {TOKENIZER_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

HEADER_FEATURES = [
    "spf_pass", "spf_fail", "spf_none",
    "dkim_pass", "dkim_fail", "dkim_none",
    "dmarc_pass", "dmarc_fail", "dmarc_none",
    "sender_domain_match",
]

def prepare_header_features(df):
    feats = np.zeros((len(df), len(HEADER_FEATURES)), dtype=np.float32)
    for i, col in enumerate(HEADER_FEATURES):
        if col in df.columns:
            feats[:, i] = pd.to_numeric(df[col], errors="coerce").fillna(0.0).values
    return feats

class NETRATier2Dataset(Dataset):
    def __init__(self, texts, header_feats, tier1_confs, labels, tokenizer, max_len=512):
        self.texts = texts
        self.header_feats = torch.tensor(header_feats, dtype=torch.float32)
        self.tier1_confs = torch.tensor(tier1_confs, dtype=torch.float32).unsqueeze(1)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "header_features": self.header_feats[idx],
            "tier1_conf": self.tier1_confs[idx],
            "label": self.labels[idx]
        }

# Filter by splits
train_df = df_tier2[df_tier2["split"] == "train"].reset_index(drop=True)
val_df   = df_tier2[df_tier2["split"] == "val"].reset_index(drop=True)
test_df  = df_tier2[df_tier2["split"] == "test"].reset_index(drop=True)

train_headers = prepare_header_features(train_df)
val_headers   = prepare_header_features(val_df)
test_headers  = prepare_header_features(test_df)

train_t1_conf = pd.to_numeric(train_df.get("tier1_confidence", 0.5), errors="coerce").fillna(0.5).values
val_t1_conf   = pd.to_numeric(val_df.get("tier1_confidence", 0.5), errors="coerce").fillna(0.5).values
test_t1_conf  = pd.to_numeric(test_df.get("tier1_confidence", 0.5), errors="coerce").fillna(0.5).values

train_dataset = NETRATier2Dataset(train_df["body_text"].fillna("").tolist(), train_headers, train_t1_conf, train_df["tier2_label"].astype(int).values, tokenizer, MAX_LENGTH)
val_dataset   = NETRATier2Dataset(val_df["body_text"].fillna("").tolist(), val_headers, val_t1_conf, val_df["tier2_label"].astype(int).values, tokenizer, MAX_LENGTH)
test_dataset  = NETRATier2Dataset(test_df["body_text"].fillna("").tolist(), test_headers, test_t1_conf, test_df["tier2_label"].astype(int).values, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, pin_memory=torch.cuda.is_available())

print(f"DataLoaders Ready: {len(train_dataset):,} train, {len(val_dataset):,} val, {len(test_dataset):,} test")


## Section 5: Model Architecture Definition

The Tier-2 classifier fuses:
1. **RoBERTa-base [CLS] vector:** 768 dimensions representing semantic meaning of the email.
2. **Header Feature Projection:** 10 dimensions → `Linear(10, 32)` → ReLU.
3. **Tier-1 Confidence:** 1 dimension.
4. **Fused Classifier:** Concatenates to 801 dimensions → `Linear(801, 256)` → ReLU → Dropout(0.3) → `Linear(256, 3)`.


In [ ]:
import torch.nn as nn
from transformers import AutoModel

class RoBERTaPhishingClassifier(nn.Module):
    def __init__(self, model_name="roberta-base", num_classes=3, dropout=0.3):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        self.header_projection = nn.Linear(10, 32)
        
        # 768 (RoBERTa CLS) + 32 (projected headers) + 1 (Tier-1 confidence) = 801
        self.classifier = nn.Sequential(
            nn.Linear(768 + 32 + 1, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask, header_features, tier1_conf):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_rep = outputs.last_hidden_state[:, 0, :]  # <s> token representation
        
        header_rep = torch.relu(self.header_projection(header_features))
        
        # Concatenate: [batch_size, 768 + 32 + 1 = 801]
        fused = torch.cat([cls_rep, header_rep, tier1_conf], dim=1)
        logits = self.classifier(fused)
        return logits

print("Instantiating RoBERTa Tier-2 Model...")
model = RoBERTaPhishingClassifier().to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")


## Section 6: Training Setup (Weighted Loss, AdamW, FP16 AMP)

- **Class-weighted CrossEntropyLoss:** Compares class frequencies to counteract class imbalance.
- **AdamW:** `learning_rate = 2e-5`, `weight_decay = 0.01`.
- **FP16 Mixed Precision:** Uses `torch.amp.GradScaler()` for 2-3x speedup and 50% VRAM reduction on RTX 5050.


In [ ]:
from transformers import get_linear_schedule_with_warmup
from sklearn.utils.class_weight import compute_class_weight

EPOCHS = 5
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

# Compute class weights
train_labels = train_df["tier2_label"].astype(int).values
classes = np.unique(train_labels)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_labels)
class_weights_tensor = torch.tensor(weights, dtype=torch.float32).to(device)
print(f"Calculated Class Weights: {weights.round(3).tolist()}")

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

# FP16 Automatic Mixed Precision Scaler
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
print(f"Total Training Steps : {total_steps:,} ({warmup_steps} warmup steps)")
print(f"AMP FP16 Enabled     : {torch.cuda.is_available()}")


## Section 7: Fine-Tuning Loop with Early Stopping

Runs training with live progress bar and validates after every epoch on Macro-F1 score.


In [ ]:
import time
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, accuracy_score

best_val_f1 = 0.0
best_epoch = 0
history = []
patience = 2
patience_counter = 0

print("=" * 65)
print("STARTING TIER-2 ROBERTA TRAINING")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_train_loss = 0.0
    start_time = time.time()
    
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    for batch in train_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        header_feats = batch["header_features"].to(device)
        tier1_conf = batch["tier1_conf"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits = model(input_ids, attention_mask, header_feats, tier1_conf)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_train_loss += loss.item()
        train_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_train_loss = total_train_loss / len(train_loader)

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    val_preds, val_targets = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            header_feats = batch["header_features"].to(device)
            tier1_conf = batch["tier1_conf"].to(device)
            labels = batch["label"].to(device)

            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                logits = model(input_ids, attention_mask, header_feats, tier1_conf)
                loss = criterion(logits, labels)

            val_loss += loss.item()
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_acc = accuracy_score(val_targets, val_preds)
    val_macro_f1 = f1_score(val_targets, val_preds, average="macro")
    elapsed = time.time() - start_time

    history.append({
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss,
        "val_acc": val_acc,
        "val_macro_f1": val_macro_f1,
        "time_sec": elapsed
    })

    print(f"Epoch {epoch:02d} Summary ({elapsed:.1f}s): "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val Acc: {val_acc*100:.2f}% | "
          f"Val Macro-F1: {val_macro_f1:.4f}")

    # Checkpoint on best Macro-F1
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), TIER2_MODEL)
        print(f"  >>> Best model saved! (Macro-F1: {val_macro_f1:.4f})")
    else:
        patience_counter += 1
        print(f"  --- No improvement (Patience: {patience_counter}/{patience})")
        if patience_counter >= patience:
            print(f"Early stopping triggered at Epoch {epoch}!")
            break

print("=" * 65)
print(f"TRAINING COMPLETE. Best Epoch: {best_epoch} with Val Macro-F1: {best_val_f1:.4f}")
print("=" * 65)


## Section 8: Final Evaluation & Confusion Matrix

Loads the best checkpoint and performs evaluation on the unseen test set.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import json

# Reload best model weights
model.load_state_dict(torch.load(TIER2_MODEL, map_location=device))
model.eval()

test_preds, test_targets = [], []
test_probs = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating on Test Set"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        header_feats = batch["header_features"].to(device)
        tier1_conf = batch["tier1_conf"].to(device)

        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits = model(input_ids, attention_mask, header_feats, tier1_conf)
            probs = torch.softmax(logits, dim=1).cpu().numpy()

        preds = np.argmax(probs, axis=1)
        test_preds.extend(preds)
        test_probs.extend(probs)
        test_targets.extend(batch["label"].numpy())

target_names = ["LEGITIMATE (0)", "SUSPICIOUS (1)", "PHISHING (2)"]
report_dict = classification_report(test_targets, test_preds, target_names=target_names, output_dict=True)
report_text = classification_report(test_targets, test_preds, target_names=target_names)
cm = confusion_matrix(test_targets, test_preds)

print("=" * 65)
print("TIER-2 TEST SET CLASSIFICATION REPORT")
print("=" * 65)
print(report_text)

print("Confusion Matrix:")
print("Predicted ->    LEGITIMATE   SUSPICIOUS   PHISHING")
for i, row in enumerate(cm):
    print(f"Actual {target_names[i][:10]:<10}: {row[0]:<12} {row[1]:<12} {row[2]:<12}")

# Save metrics JSON
metrics_payload = {
    "test_accuracy": report_dict["accuracy"],
    "macro_f1": report_dict["macro avg"]["f1-score"],
    "weighted_f1": report_dict["weighted avg"]["f1-score"],
    "per_class": {
        "legitimate": report_dict["LEGITIMATE (0)"],
        "suspicious": report_dict["SUSPICIOUS (1)"],
        "phishing": report_dict["PHISHING (2)"]
    },
    "confusion_matrix": cm.tolist(),
    "best_epoch": best_epoch,
    "best_val_macro_f1": best_val_f1
}

with open(TIER2_METRICS, "w") as f:
    json.dump(metrics_payload, f, indent=2)

print(f"\nSaved evaluation metrics to: {TIER2_METRICS}")


## Section 9: Explainable AI (XAI) with Integrated Gradients

Tier-2 implements token attribution via `captum` Integrated Gradients. This highlights the exact words driving the escalation verdict.


In [ ]:
try:
    from captum.attr import LayerIntegratedGradients
    
    # Target RoBERTa word embeddings layer
    lig = LayerIntegratedGradients(
        lambda ids, mask, hf, t1: model(ids, mask, hf, t1),
        model.roberta.embeddings.word_embeddings
    )
    
    # Pick a sample phishing test email
    sample_idx = 0
    for idx, (t, p) in enumerate(zip(test_targets, test_preds)):
        if t == 2 and p == 2:
            sample_idx = idx
            break
            
    sample_text = str(test_df.iloc[sample_idx]["body_text"])[:300]
    enc = tokenizer(sample_text, max_length=128, padding="max_length", truncation=True, return_tensors="pt")
    
    ids = enc["input_ids"].to(device)
    mask = enc["attention_mask"].to(device)
    hf = test_headers[sample_idx:sample_idx+1]
    hf = torch.tensor(hf, dtype=torch.float32).to(device)
    t1 = torch.tensor([[test_t1_conf[sample_idx]]], dtype=torch.float32).to(device)
    
    attributions, delta = lig.attribute(
        inputs=ids,
        additional_forward_args=(mask, hf, t1),
        target=2, # Target class: PHISHING
        return_convergence_delta=True
    )
    
    # Aggregate token importance
    tokens = tokenizer.convert_ids_to_tokens(ids[0])
    token_scores = attributions.sum(dim=-1).squeeze(0).detach().cpu().numpy()
    
    print("=" * 65)
    print("INTEGRATED GRADIENTS TOKEN ATTRIBUTIONS (Top Phishing Cues)")
    print("=" * 65)
    top_indices = np.argsort(token_scores)[::-1][:15]
    for rank, idx in enumerate(top_indices, 1):
        tok = tokens[idx].replace("Ġ", "")
        if tok not in ["<s>", "</s>", "<pad>", "", " "]:
            print(f"  {rank:2d}. {tok:<15} (Attribution Score: {token_scores[idx]:+.4f})")

except ImportError:
    print("Captum not installed in current environment. Install via: pip install captum")


## Section 10: Export Artifacts & Verification

Saves the tokenizer and inference configuration matching the Tier-2 FastAPI service specifications.


In [ ]:
# 1. Save Tokenizer
print(f"Saving RoBERTa tokenizer to {TIER2_TOK}...")
tokenizer.save_pretrained(str(TIER2_TOK))

# 2. Save Inference Config
tier2_config_data = {
    "model_name": "roberta-base",
    "num_classes": 3,
    "max_length": MAX_LENGTH,
    "class_names": ["LEGITIMATE", "SUSPICIOUS", "PHISHING"],
    "header_features": HEADER_FEATURES,
    "thresholds": {
        "suspicious_lower": 0.35,
        "phishing_lower": 0.65
    },
    "metrics": {
        "test_accuracy": metrics_payload["test_accuracy"],
        "macro_f1": metrics_payload["macro_f1"]
    }
}

with open(TIER2_CONFIG, "w") as f:
    json.dump(tier2_config_data, f, indent=2)

# Also save general tier2_config.json in models dir
with open(MODELS_DIR / "tier2_config.json", "w") as f:
    json.dump(tier2_config_data, f, indent=2)

print("\n" + "=" * 65)
print("ALL TIER-2 ARTIFACTS SAVED SUCCESSFULLY!")
print("=" * 65)
print(f"1. Model Weights   : {TIER2_MODEL} ({TIER2_MODEL.stat().st_size / 1e6:.1f} MB)")
print(f"2. Tokenizer Dir   : {TIER2_TOK}")
print(f"3. Config File     : {TIER2_CONFIG}")
print(f"4. Metrics File    : {TIER2_METRICS}")


## Section 11: Summary & Next Steps for Ramanan

Congratulations! Tier-2 RoBERTa training is complete.

### What to do next:
1. **Transfer the weights to your main laptop:**
   - Transfer `ml/models/roberta_tier2.pt` (~498 MB)
   - Transfer `ml/models/roberta_tier2_tokenizer/`
   - Transfer `ml/models/roberta_tier2_config.json`
   - Transfer `ml/evaluation/roberta_tier2_metrics.json`
   - *(You can use PairDrop, USB, or Google Drive for this)*

2. **Run Cloud Run Containerization / Test Service:**
   ```bash
   # Run the Tier-2 inference service locally:
   uvicorn api.tier2_service.main:app --port 8001
   
   # Or run Tier-1 which automatically connects to Tier-2:
   uvicorn api.main:app --reload
   ```

3. **Deploy Tier-2 Service to GCP Cloud Run:**
   - Build with the included `api/tier2_service/Dockerfile`.
   - Set `TIER2_SERVICE_URL` in your `.env` file to your Cloud Run URL.
